In [2]:
import pandas as pd
import glob
import os
import re

# 1. 환경 설정
# 데이터 폴더들이 있는 경로를 지정합니다. (현재 폴더면 '.' 유지)
root_path = 'C:/Users/user/Desktop/semi2/주제별 일상 대화 데이터세트' 

# 2. 모든 하위 폴더에서 .txt 파일 목록 가져오기
# 'TS_'로 시작하는 폴더 내부의 모든 .txt 파일을 찾습니다.
# recursive=True를 통해 하위 폴더 깊숙이 있는 파일까지 모두 검색합니다.
file_pattern = os.path.join(root_path, 'TS_*', '*.txt')
file_list = glob.glob(file_pattern)

print(f"총 {len(file_list)}개의 파일을 찾았습니다. 추출을 시작합니다...")

all_utterances = []

# 3. 파일 순회 및 텍스트 추출
for file_path in file_list:
    try:
        # 파일 열기 (인코딩 에러 방지를 위해 utf-8 사용)
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                # 데이터 형식: "1 : 문장", "2 : 문장"
                # 정규표현식을 사용하여 숫자와 콜론( : ) 뒤의 실제 문장만 추출합니다.
                match = re.search(r'^\d+\s:\s(.*)', line)
                if match:
                    utterance = match.group(1).strip()
                    if utterance:  # 빈 문장이 아니면 리스트에 추가
                        all_utterances.append(utterance)
    except Exception as e:
        print(f"파일 읽기 오류 ({os.path.basename(file_path)}): {e}")

# 4. 데이터프레임 생성
# 논문 설계에 따라 모든 발화의 인텐트는 '일상'입니다.
df_subject = pd.DataFrame({'utterance': all_utterances, 'intent': '일상'})

# 5. 데이터 정제 (선택 사항)
# 중복 발화 제거 및 인덱스 초기화
initial_count = len(df_subject)
df_subject = df_subject.drop_duplicates().reset_index(drop=True)
print(f"중복 제거 전: {initial_count}건 -> 중복 제거 후: {len(df_subject)}건")

# 6. 결과 저장
# 한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용
df_subject.to_csv('pre_subject_daily.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ 주제별 일상 대화 전처리 완료!")
print(f"최종 저장된 파일: pre_subject_daily.csv")
print(f"추출된 총 문장 수: {len(df_subject)}건")

총 87690개의 파일을 찾았습니다. 추출을 시작합니다...
중복 제거 전: 1436948건 -> 중복 제거 후: 1375251건

✅ 주제별 일상 대화 전처리 완료!
최종 저장된 파일: pre_subject_daily.csv
추출된 총 문장 수: 1375251건
